# S6E8 - Submission Blends
## 8 highest-EV blends of best public LB files
### Blend 1:.96976
### Blend 2:.96978
### Blend 3:.96973
### Blend 4:.96961
### Blend 5:.96975
### Blend 6:.96977
### Blend 7:.96967
### Blend 8:.96968


## Setup

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit, logit

OUT_DIR = Path("blends")
OUT_DIR.mkdir(exist_ok=True)
TARGET = "addicted_label"
EPS = 1e-6


## Select Members

In [15]:
MEMBERS = [
    ("score96980.csv", 0.96980),
    ("score96973.csv", 0.96973),
    ("score96965.csv", 0.96965),
    ("score96952.csv", 0.96952),
    ("score96951.csv", 0.96951),
    ("score96941.csv", 0.96941),
    ("score96937.csv", 0.96937),
    ("score96918.csv", 0.96918),
]

frames = []
scores = []
names = []
for path, lb in MEMBERS:
    df = pd.read_csv(path).sort_values("id").reset_index(drop=True)
    if frames:
        assert (df["id"].to_numpy() == frames[0]["id"].to_numpy()).all()
    frames.append(df)
    scores.append(lb)
    names.append(Path(path).stem)

ids = frames[0]["id"].to_numpy()
P = np.column_stack([f[TARGET].to_numpy(dtype=np.float64) for f in frames])
scores = np.asarray(scores, dtype=np.float64)
best = P[:, 0]
print("members", len(names), "rows", len(ids), "P", P.shape)
print(list(zip(names, scores)))
pd.DataFrame(P, columns=names).describe().T


members 8 rows 296302 P (296302, 8)
[('score96980', np.float64(0.9698)), ('score96973', np.float64(0.96973)), ('score96965', np.float64(0.96965)), ('score96952', np.float64(0.96952)), ('score96951', np.float64(0.96951)), ('score96941', np.float64(0.96941)), ('score96937', np.float64(0.96937)), ('score96918', np.float64(0.96918))]


,count,mean,std,min,25%,50%,75%,max
score96980,296302.0,0.691036,0.385412,0.000115,0.303553,0.948418,0.999030,0.999999
score96973,296302.0,0.689372,0.385861,0.000130,0.299370,0.946459,0.998943,0.999999
score96965,296302.0,0.689755,0.385488,0.000152,0.300667,0.946392,0.998913,0.999999
score96952,296302.0,0.688227,0.386244,0.000145,0.295935,0.945369,0.998845,0.999999
score96951,296302.0,0.689223,0.384906,0.000116,0.301104,0.944665,0.998885,0.999999
score96941,296302.0,0.690409,0.385228,0.000136,0.302944,0.946663,0.999064,0.999999
score96937,296302.0,0.690812,0.384906,0.000150,0.304421,0.946554,0.998990,0.999999
score96918,296302.0,0.690392,0.384264,0.000122,0.305584,0.945269,0.998861,0.999999


## Correlation

In [16]:
corr = pd.DataFrame(P, columns=names).corr()
print(corr.round(5))
print(
    "mean off-diag corr",
    ((corr.values.sum() - len(names)) / (len(names) * (len(names) - 1))).round(5),
)


            score96980  score96973  score96965  score96952  score96951  \
score96980     1.00000     0.99973     0.99970     0.99950     0.99947   
score96973     0.99973     1.00000     0.99989     0.99976     0.99972   
score96965     0.99970     0.99989     1.00000     0.99981     0.99975   
score96952     0.99950     0.99976     0.99981     1.00000     0.99983   
score96951     0.99947     0.99972     0.99975     0.99983     1.00000   
score96941     0.99933     0.99952     0.99957     0.99961     0.99956   
score96937     0.99931     0.99951     0.99957     0.99963     0.99960   
score96918     0.99821     0.99842     0.99846     0.99855     0.99854   

            score96941  score96937  score96918  
score96980     0.99933     0.99931     0.99821  
score96973     0.99952     0.99951     0.99842  
score96965     0.99957     0.99957     0.99846  
score96952     0.99961     0.99963     0.99855  
score96951     0.99956     0.99960     0.99854  
score96941     1.00000     0.99955     

## Blend Helpers

In [17]:
def clip01(x):
    return np.clip(x, 0.0, 1.0)


def write_blend(tag, preds, note=""):
    preds = clip01(np.asarray(preds, dtype=np.float64))
    out = pd.DataFrame({"id": ids, TARGET: preds})
    path = OUT_DIR / f"{tag}.csv"
    out.to_csv(path, index=False)
    print(
        f"{tag}: mean={preds.mean():.5f} std={preds.std():.5f} "
        f"min={preds.min():.5f} max={preds.max():.5f}  {note}"
    )
    print(" ->", path)
    return out


def normalize(w):
    w = np.clip(np.asarray(w, dtype=np.float64), 0.0, None)
    s = w.sum()
    if s <= 0:
        return np.ones(len(w)) / len(w)
    return w / s


def lb_gap_weights(power=10.0, floor=0.96800):
    gap = np.maximum(scores - floor, 1e-6)
    return normalize(gap ** power)


ranks = np.zeros_like(P)
for j in range(P.shape[1]):
    ranks[:, j] = pd.Series(P[:, j]).rank(method="average").to_numpy()
rank_avg = (ranks.mean(axis=1) - 1.0) / (len(ids) - 1.0)


## 1 Best + Rank 55/45

In [18]:
b1 = 0.55 * best + 0.45 * rank_avg
write_blend("blend01_best_rank_55", b1, "0.55*96980 + 0.45*rank_avg")


blend01_best_rank_55: mean=0.60507 std=0.33361 min=0.00006 max=1.00000  0.55*96980 + 0.45*rank_avg
 -> blends\blend01_best_rank_55.csv


,id,addicted_label
0,691369,0.912701
1,691370,0.769360
2,691371,0.627683
3,691372,0.824591
4,691373,0.871163
...,...,...
296297,987666,0.994664
296298,987667,0.544178
296299,987668,0.164250
296300,987669,0.539497


## 2 Best + Rank 70/30

In [19]:
b2 = 0.70 * best + 0.30 * rank_avg
write_blend("blend02_best_rank_70", b2, "0.70*96980 + 0.30*rank_avg")


blend02_best_rank_70: mean=0.63373 std=0.34968 min=0.00008 max=1.00000  0.70*96980 + 0.30*rank_avg
 -> blends\blend02_best_rank_70.csv


,id,addicted_label
0,691369,0.941680
1,691370,0.834487
2,691371,0.684469
3,691372,0.880259
4,691373,0.913352
...,...,...
296297,987666,0.996439
296298,987667,0.595325
296299,987668,0.157829
296300,987669,0.588234


## 3 LB Power Weights

In [20]:
w3 = lb_gap_weights(power=10.0, floor=0.96800)
b3 = P @ w3
write_blend("blend03_lb_power", b3, f"weights={np.round(w3, 4).tolist()}")


blend03_lb_power: mean=0.69006 std=0.38549 min=0.00013 max=1.00000  weights=[0.3824, 0.2572, 0.1602, 0.0705, 0.066, 0.0333, 0.0249, 0.0056]
 -> blends\blend03_lb_power.csv


,id,addicted_label
0,691369,0.999560
1,691370,0.966129
2,691371,0.814757
3,691372,0.992269
4,691373,0.998095
...,...,...
296297,987666,0.999987
296298,987667,0.683423
296299,987668,0.143680
296300,987669,0.677039


## 4 Rank Average

In [21]:
b4 = rank_avg
write_blend("blend04_rank_avg", b4, "mean rank / (n-1) over all 8")


blend04_rank_avg: mean=0.50000 std=0.28858 min=0.00000 max=1.00000  mean rank / (n-1) over all 8
 -> blends\blend04_rank_avg.csv


,id,addicted_label
0,691369,0.806442
1,691370,0.530564
2,691371,0.419468
3,691372,0.620473
4,691373,0.716473
...,...,...
296297,987666,0.988158
296298,987667,0.356641
296299,987668,0.187796
296300,987669,0.360795


## 5 Greedy Caruana + Softmax LB

In [22]:
def caruana_select(max_members=5):
    chosen = [0]
    remaining = list(range(1, P.shape[1]))
    while remaining and len(chosen) < max_members:
        best_j = None
        best_score = -1e9
        for j in remaining:
            trial = chosen + [j]
            sub = P[:, trial]
            quality = scores[trial].mean()
            redun = np.corrcoef(sub.T)
            if redun.ndim == 0:
                avg_corr = 0.0
            else:
                m = len(trial)
                avg_corr = (redun.sum() - m) / (m * (m - 1))
            score = quality - 0.002 * avg_corr
            if score > best_score:
                best_score = score
                best_j = j
        chosen.append(best_j)
        remaining.remove(best_j)
    return chosen


sel = caruana_select(max_members=5)
print("selected", [names[i] for i in sel], [scores[i] for i in sel])
gap_sel = np.maximum(scores[sel] - 0.96800, 1e-6)
temp = 0.00015
logits_w = gap_sel / temp
logits_w = logits_w - logits_w.max()
w5 = np.exp(logits_w)
w5 = w5 / w5.sum()
b5 = P[:, sel] @ w5
write_blend(
    "blend05_caruana_softmax",
    b5,
    f"sel={[names[i] for i in sel]} w={np.round(w5, 4).tolist()}",
)


selected ['score96980', 'score96973', 'score96965', 'score96952', 'score96951'] [np.float64(0.9698), np.float64(0.96973), np.float64(0.96965), np.float64(0.96952), np.float64(0.96951)]
blend05_caruana_softmax: mean=0.69007 std=0.38553 min=0.00013 max=1.00000  sel=['score96980', 'score96973', 'score96965', 'score96952', 'score96951'] w=[0.4359, 0.2733, 0.1603, 0.0674, 0.0631]
 -> blends\blend05_caruana_softmax.csv


,id,addicted_label
0,691369,0.999566
1,691370,0.965948
2,691371,0.811606
3,691372,0.992219
4,691373,0.998054
...,...,...
296297,987666,0.999987
296298,987667,0.685500
296299,987668,0.143694
296300,987669,0.676565


## 6 Best + LB Power

In [23]:
b6 = 0.50 * best + 0.50 * b3
write_blend("blend06_best_lb_power", b6, "0.50*96980 + 0.50*lb_power")


blend06_best_lb_power: mean=0.69055 std=0.38544 min=0.00012 max=1.00000  0.50*96980 + 0.50*lb_power
 -> blends\blend06_best_lb_power.csv


,id,addicted_label
0,691369,0.999600
1,691370,0.965434
2,691371,0.806399
3,691372,0.991933
4,691373,0.997912
...,...,...
296297,987666,0.999987
296298,987667,0.690520
296299,987668,0.144333
296300,987669,0.681373


## 7 Logit Mean Top5

In [24]:
Pc5 = np.clip(P[:, :5], EPS, 1.0 - EPS)
b7 = expit(logit(Pc5).mean(axis=1))
write_blend("blend07_logit_top5", b7, "sigmoid(mean(logit(p))) top 5 LB")


blend07_logit_top5: mean=0.68954 std=0.38558 min=0.00013 max=1.00000  sigmoid(mean(logit(p))) top 5 LB
 -> blends\blend07_logit_top5.csv


,id,addicted_label
0,691369,0.999528
1,691370,0.965937
2,691371,0.824476
3,691372,0.992316
4,691373,0.998195
...,...,...
296297,987666,0.999988
296298,987667,0.671117
296299,987668,0.143525
296300,987669,0.673978


## 8 Diversity × LB Weights

In [25]:
C = np.corrcoef(P.T)
mean_corr = (C.sum(axis=1) - 1.0) / (len(names) - 1)
diversity = np.maximum(1.0 - mean_corr, 1e-6)
gap = np.maximum(scores - 0.96800, 1e-6)
w8 = normalize((gap ** 6.0) * (diversity ** 2.0))
b8 = P @ w8
write_blend(
    "blend08_div_lb",
    b8,
    f"mean_corr={np.round(mean_corr, 4).tolist()} w={np.round(w8, 4).tolist()}",
)


blend08_div_lb: mean=0.69022 std=0.38516 min=0.00013 max=1.00000  mean_corr=[0.9993, 0.9995, 0.9995, 0.9995, 0.9995, 0.9994, 0.9994, 0.9984] w=[0.3483, 0.1431, 0.0961, 0.0612, 0.067, 0.0725, 0.0604, 0.1514]
 -> blends\blend08_div_lb.csv


,id,addicted_label
0,691369,0.999560
1,691370,0.966100
2,691371,0.821112
3,691372,0.992248
4,691373,0.998108
...,...,...
296297,987666,0.999988
296298,987667,0.675842
296299,987668,0.143493
296300,987669,0.680980


## Export

In [26]:
recommended = {
    "blend01_best_rank_55": b1,
    "blend02_best_rank_70": b2,
    "blend03_lb_power": b3,
    "blend04_rank_avg": b4,
    "blend05_caruana_softmax": b5,
    "blend06_best_lb_power": b6,
    "blend07_logit_top5": b7,
    "blend08_div_lb": b8,
}

pd.DataFrame({"id": ids, TARGET: clip01(b1)}).to_csv("submission.csv", index=False)
print("submission.csv <- blend01_best_rank_55")

summary = []
for tag, preds in recommended.items():
    summary.append(
        {
            "file": f"blends/{tag}.csv",
            "mean": float(np.mean(preds)),
            "std": float(np.std(preds)),
            "corr_to_best": float(np.corrcoef(preds, best)[0, 1]),
        }
    )
pd.DataFrame(summary)


submission.csv <- blend01_best_rank_55


,file,mean,std,corr_to_best
0,blends/blend01_best_rank_55.csv,0.605070,0.333611,0.985371
1,blends/blend02_best_rank_70.csv,0.633725,0.349685,0.994108
2,blends/blend03_lb_power.csv,0.690055,0.385489,0.999893
3,blends/blend04_rank_avg.csv,0.500000,0.288581,0.899069
4,blends/blend05_caruana_softmax.csv,0.690072,0.385532,0.999913
5,blends/blend06_best_lb_power.csv,0.690546,0.385440,0.999973
6,blends/blend07_logit_top5.csv,0.689537,0.385583,0.999794
7,blends/blend08_div_lb.csv,0.690225,0.385158,0.999829
